-------------------------------
-------------------------------
# Laboratorio #10 - IA (CC3085)
* Dulce Ambrosio - 231143
* Daniel Chet - 231177
* Gadiel Ocaña - 231270

-------------------------------
-------------------------------

-------------------------------
**Task 2:** Depuración de un sistema con bug deliberado

-------------------------------

In [ ]:
# logtrack_buggy.py
import numpy as np

CARRILES = 20
K = 10

def transicion(h_prev):
    # El vehiculo se mueve +1, 0 o -1 carril con igual probabilidad
    delta = np.random.choice([-1, 0, 1])
    return int(np.clip(h_prev + delta, 0, CARRILES - 1))

def emision(sensor, h):
    # Sensor reporta el carril real con prob 0.6, adyacente con 0.2, error con 0.2
    dist = abs(sensor - h)
    if dist == 0: return 0.6
    elif dist == 1: return 0.2
    else: return 0.2 / (CARRILES - 2)

# ────────────────────────────────────────────────────────────
# VERSIÓN BUGGY (Dado en el documento con la misma semilla)
# ────────────────────────────────────────────────────────────
def filtrado_particulas_buggy(observaciones, seed=42):
    np.random.seed(seed)
    particulas = np.random.randint(0, CARRILES, K)
    print(f"{'t':<5} {'Sensor':<8} {'Partículas':<45} {'Media'}")
    print("-" * 70)
    for t, sensor in enumerate(observaciones):
        # PASO 1: Proponer
        propuestas = np.array([transicion(h) for h in particulas])
        # PASO 2: Ponderar
        pesos = np.array([emision(sensor, h) for h in propuestas])
        pesos_norm = pesos / pesos.sum()
        # PASO 3: Remuestrear <-- REVISEN ESTA LINEA
        idx = np.argsort(pesos_norm)[-K:] # BUG: Selecciona las K partículas con mayor peso, no proporcional al peso
        particulas = propuestas[idx]
        print(f"t={t+1:<3} {sensor:<8} {str(sorted(particulas.tolist())):<45} {np.mean(particulas):.2f}")
    return particulas


# ─────────────────────────────────────────────
# VERSIÓN CORREGIDA — Con la misma semilla
# ─────────────────────────────────────────────
def filtrado_particulas_corregido(observaciones, seed=42):
    np.random.seed(seed)
    particulas = np.random.randint(0, CARRILES, K)
    print(f"{'t':<5} {'Sensor':<8} {'Partículas':<45} {'Media'}")
    print("-" * 70)
    for t, sensor in enumerate(observaciones):
        # PASO 1: Proponer 
        propuestas = np.array([transicion(h) for h in particulas])
        # PASO 2: Ponderar
        pesos = np.array([emision(sensor, h) for h in propuestas])
        pesos_norm = pesos / pesos.sum()
        # PASO 3 (CORRECCIÓN): remuestreo estocástico proporcional al peso
        # Cada partícula i es seleccionada con probabilidad pesos_norm[i]
        idx = np.random.choice(len(propuestas), size=K, replace=True, p=pesos_norm)
        particulas = propuestas[idx]
        print(f"t={t+1:<3} {sensor:<8} {str(sorted(particulas.tolist())):<45} {np.mean(particulas):.2f}")
    return particulas

# Secuencia de sensores simulando movimiento real del vehiculo
observaciones_orig = [5, 6, 7, 7, 8, 8, 3, 4, 5]

print("=" * 70)
print("BUGGY (Beam Search) — seed=42")
print("=" * 70)
filtrado_particulas_buggy(observaciones_orig, seed=42)

print()
print("=" * 70)
print("CORREGIDO (Particle Filter) — seed=42")
print("=" * 70)
filtrado_particulas_corregido(observaciones_orig, seed=42)

BUGGY (Beam Search) — seed=42
t     Sensor   Partículas                                    Media
----------------------------------------------------------------------
t=1   5        [3, 6, 7, 7, 9, 9, 10, 13, 18, 19]            10.10
t=2   6        [4, 6, 7, 8, 9, 9, 10, 12, 18, 19]            10.20
t=3   7        [3, 7, 8, 9, 9, 10, 11, 13, 18, 18]           10.60
t=4   7        [3, 7, 7, 9, 10, 10, 10, 12, 17, 18]          10.30
t=5   8        [3, 6, 7, 9, 10, 11, 11, 12, 16, 19]          10.40
t=6   8        [2, 7, 7, 9, 10, 11, 11, 12, 16, 19]          10.40
t=7   3        [2, 7, 8, 8, 10, 11, 12, 12, 16, 19]          10.50
t=8   4        [1, 6, 7, 7, 11, 11, 11, 12, 15, 18]          9.90
t=9   5        [2, 6, 7, 8, 10, 10, 10, 11, 16, 19]          9.90

CORREGIDO (Particle Filter) — seed=42
t     Sensor   Partículas                                    Media
----------------------------------------------------------------------
t=1   5        [6, 6, 6, 6, 6, 6, 6, 6, 6, 10]        

array([5, 6, 5, 6, 5, 6, 5, 6, 5, 5])

In [2]:
# ──────────────────────────────────────────────────
# SECUENCIA DE OBSERVACIONES PARA QUE EL BUG FALLE
# ──────────────────────────────────────────────────
secuencia_critica = [16, 16, 16, 16, 16, 3, 3, 3, 3]
pos_real         = [16, 16, 16, 16, 16, 3, 3, 3, 3]

print("=" * 70)
print("BUGGY — secuencia crítica, seed=40")
print("=" * 70)
filtrado_particulas_buggy(secuencia_critica, seed=40)

print()
print("=" * 70)
print("CORREGIDO — secuencia crítica, seed=40")
print("=" * 70)
filtrado_particulas_corregido(secuencia_critica, seed=40)

BUGGY — secuencia crítica, seed=40
t     Sensor   Partículas                                    Media
----------------------------------------------------------------------
t=1   16       [1, 5, 6, 6, 6, 9, 11, 18, 19, 19]            10.00
t=2   16       [2, 5, 5, 6, 6, 9, 11, 18, 19, 19]            10.00
t=3   16       [1, 6, 6, 7, 7, 10, 10, 18, 19, 19]           10.30
t=4   16       [1, 5, 7, 7, 7, 9, 10, 17, 18, 19]            10.00
t=5   16       [2, 5, 7, 8, 8, 9, 9, 17, 19, 19]             10.30
t=6   3        [1, 4, 7, 8, 8, 9, 9, 17, 18, 18]             9.90
t=7   3        [2, 3, 6, 7, 9, 9, 10, 18, 18, 19]            10.10
t=8   3        [2, 3, 7, 7, 8, 10, 11, 18, 19, 19]           10.40
t=9   3        [1, 2, 6, 8, 9, 9, 12, 19, 19, 19]            10.40

CORREGIDO — secuencia crítica, seed=40
t     Sensor   Partículas                                    Media
----------------------------------------------------------------------
t=1   16       [1, 1, 5, 5, 6, 6, 9, 11, 19, 19

array([3, 3, 3, 4, 3, 3, 3, 3, 4, 4])

In [3]:
# Tabla resumen de errores
def get_medias(observaciones, seed, modo):
    np.random.seed(seed)
    particulas = np.random.randint(0, CARRILES, K)
    medias = []
    for sensor in observaciones:
        propuestas = np.array([transicion(h) for h in particulas])
        pesos = np.array([emision(sensor, h) for h in propuestas])
        pesos_norm = pesos / pesos.sum()
        if modo == 'buggy':
            idx = np.argsort(pesos_norm)[-K:]
        else:
            idx = np.random.choice(len(propuestas), size=K, replace=True, p=pesos_norm)
        particulas = propuestas[idx]
        medias.append(np.mean(particulas))
    return medias

mb = get_medias(secuencia_critica, 40, 'buggy')
mc = get_medias(secuencia_critica, 40, 'corregido')

print(f"{'t':<5} {'Real':<6} {'BUGGY media':<14} {'Error B':<12} {'CORR. media':<14} {'Error C'}")
print("-" * 62)
for t, real in enumerate(pos_real):
    eb = abs(mb[t] - real)
    ec = abs(mc[t] - real)
    marca = "  ← SALTO" if t == 5 else ""
    print(f"t={t+1:<3} {real:<6} {mb[t]:<14.2f} {eb:<12.2f} {mc[t]:<14.2f} {ec:.2f}{marca}")

t     Real   BUGGY media    Error B      CORR. media    Error C
--------------------------------------------------------------
t=1   16     10.00          6.00         8.20           7.80
t=2   16     10.00          6.00         6.40           9.60
t=3   16     10.30          5.70         4.00           12.00
t=4   16     10.00          6.00         7.60           8.40
t=5   16     10.30          5.70         13.90          2.10
t=6   3      9.90           6.90         6.10           3.10  ← SALTO
t=7   3      10.10          7.10         3.40           0.40
t=8   3      10.40          7.40         2.80           0.20
t=9   3      10.40          7.40         3.30           0.30


In [4]:
# Verificación: partículas pares, sensor en carril 19 (solo carril 18 adyacente)
# Si ninguna partícula está en 18 ni 19 -> pesos uniformes

particulas_pares = np.array([0, 2, 4, 6, 8, 10, 12, 14, 16, 18])
sensor_test = 19   # Solo carril 18 es adyacente

pesos_test = np.array([emision(sensor_test, h) for h in particulas_pares])
pesos_norm_test = pesos_test / pesos_test.sum()

print("Verificación de pesos uniformes cuando ninguna partícula está en el carril exacto:")
print(f"Partículas  : {particulas_pares.tolist()}")
print(f"Sensor      : {sensor_test}")
print(f"Pesos norm. : {[round(p, 4) for p in pesos_norm_test]}")
print()
# Caso de pesos verdaderamente uniformes: sensor lejano de todas las particulas
particulas_lejanas = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])  # todas en zona 0-9
sensor_lejano = 19  # todas a distancia >1
pesos_lejanos = np.array([emision(sensor_lejano, h) for h in particulas_lejanas])
pesos_norm_lejanos = pesos_lejanos / pesos_lejanos.sum()
print("Caso con pesos perfectamente uniformes (sensor completamente lejano):")
print(f"Partículas  : {particulas_lejanas.tolist()}")
print(f"Sensor      : {sensor_lejano}")
print(f"Pesos norm. : {[round(p, 4) for p in pesos_norm_lejanos]}")
print(f"¿Uniformes? : {np.allclose(pesos_norm_lejanos, pesos_norm_lejanos[0])}")
print("\n→ En este caso buggy y corregido son estadísticamente equivalentes.")

Verificación de pesos uniformes cuando ninguna partícula está en el carril exacto:
Partículas  : [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
Sensor      : 19
Pesos norm. : [np.float64(0.037), np.float64(0.037), np.float64(0.037), np.float64(0.037), np.float64(0.037), np.float64(0.037), np.float64(0.037), np.float64(0.037), np.float64(0.037), np.float64(0.6667)]

Caso con pesos perfectamente uniformes (sensor completamente lejano):
Partículas  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Sensor      : 19
Pesos norm. : [np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1)]
¿Uniformes? : True

→ En este caso buggy y corregido son estadísticamente equivalentes.


-------------------------------
**Task 3:** Implementación y Dictamen Ejecutivo

-------------------------------